In [ ]:
pip install ragas langchain langchain-core

import os
from typing import List, Dict, Any
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_recall,     # Retriever 평가: 정답 대비 검색된 문서의 커버율 (Recall@K 개념)
    context_precision,  # Retriever 평가: 검색된 문서 중 실제 정답 문서가 상위에 잘 정렬되었는지
    faithfulness,       # Generator 평가: 할루시네이션 방지 (주어진 문서에만 기반해 답했는지)
    answer_relevance    # Generator 평가: 사용자의 질문 의도에 맞게 답변했는지
)
from langchain_community.llms import HuggingFacePipeline
from langchain_core.retrievers import BaseRetriever

def evaluate_rag_system(
    test_dataset: List[Dict[str, Any]], 
    retriever: BaseRetriever, 
    rag_chain: Any,
    evaluation_llm: HuggingFacePipeline
) -> Dict[str, float]:
    """
    RAG 시스템의 검색 및 생성 성능을 Ragas 프레임워크로 종합 평가하는 함수.

    Args:
        test_dataset: [{"query": "질문", "ground_truth": "실제 정답"}] 형태의 리스트
        retriever: LangChain Retriever 객체 (FAISS + Reranker 파이프라인)
        rag_chain: 질문을 입력받아 최종 답변 문자열을 반환하는 LangChain 실행 체인
        evaluation_llm: 평가에 사용할 로컬 LLM (Qwen2) 객체
        
    Returns:
        각 평가지표별 평균 점수가 담긴 딕셔너리
    """
    
    # Ragas 데이터 변환용 빈 리스트 초기화
    queries = []
    contexts = []
    answers = []
    ground_truths = []
    
    print("🚀 RAG 파이프라인 데이터 수집 및 추론 시작...")
    
    for idx, item in enumerate(test_dataset):
        query = item.get("query")
        gt = item.get("ground_truth")
        
        # 1. 1차+2차 검색 파이프라인 통과 (Context 추출)
        retrieved_docs = retriever.invoke(query)
        # Ragas는 각 문서 내용을 문자열 리스트 포맷으로 요구함
        context_list = [doc.page_content for doc in retrieved_docs]
        
        # 2. LLM 추론 결과 획득 (Answer 생성)
        # 생성 결과물 문자열만 깔끔하게 추출하도록 연동 필요
        generated_answer = rag_chain.invoke({"context": "\n\n".join(context_list), "question": query})
        
        # 데이터셋 적재
        queries.append(query)
        contexts.append(context_list)
        answers.append(generated_answer)
        ground_truths.append(gt)
        
        print(f"   [{idx + 1}/{len(test_dataset)}] 전처리 완료 (질문: {query[:15]}...)")

    # 3. Ragas 입력용 HuggingFace Dataset 객체 생성
    data_dict = {
        "question": queries,
        "contexts": contexts,
        "answer": answers,
        "ground_truth": ground_truths
    }
    dataset = Dataset.from_dict(data_dict)
    
    # 4. 평가에 활용할 메트릭 및 평가 LLM 셋팅
    metrics = [context_recall, context_precision, faithfulness, answer_relevance]
    
    # Ragas 기본 매트릭스들이 내부 판별을 위해 주입받은 로컬 Qwen2 모델을 활용하도록 지정
    for metric in metrics:
        metric.llm = evaluation_llm
        
    print("\n📊 Ragas 평가지표 연산 시작 (LLM 심사 진행 중)...")
    
    # 5. Ragas 평가 실행
    result = evaluate(
        dataset=dataset,
        metrics=metrics
    )
    
    return dict(result)

# ------------------------------------------------------------------------
# 사용 예시 (이 모듈을 다른 파일에서 불러와 실행할 때의 스크립트 구조)
# ------------------------------------------------------------------------
if __name__ == "__main__":
    # 1. 사내 구축된 골든 데이터셋 준비 (질문 - 정답 셋)
    sample_test_set = [
        {
            "query": "이번 사내 전산망 고도화 사업의 장애 조치 SLA 기준은 어떻게 되나요?",
            "ground_truth": "장애 발생 시 30분 이내 원인 파악, 2시간 이내에 서비스를 정상 복구해야 하며, 이를 위반 시 페널티가 부과됩니다."
        },
        {
            "query": "RFP 14페이지에 명시된 데이터 백업 주기가 궁금합니다.",
            "ground_truth": "운영 데이터베이스는 매일 자정에 증분 백업을 수행하고, 매주 일요일에 전체 백업을 수행해야 합니다."
        }
    ]
    
    # 2. 기존 파이프라인에서 정의한 객체들 배치 (이전 단계의 변수들 가정)
    # final_retriever = compression_retriever (앞서 정의한 FAISS + 리랭커 조립체)
    # my_rag_chain = chain (Qwen2 프롬프트 템플릿 체인)
    # qwen_llm = llm (평가에 재사용할 로컬 Qwen2 인스턴스)
    
    # 3. 함수 호출 테스트 (구동환경에 맞게 인자 주입)
    # scores = evaluate_rag_system(
    #     test_dataset=sample_test_set,
    #     retriever=final_retriever,
    #     rag_chain=my_rag_chain,
    #     evaluation_llm=qwen_llm
    # )
    # print("\n=======  최종 Ragas 검증 결과 =======")
    # print(scores)

In [ ]:
pip install ragas langchain langchain-core

import os
from typing import List, Dict, Any
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_recall,     # Retriever 평가: 정답 대비 검색된 문서의 커버율 (Recall@K 개념)
    context_precision,  # Retriever 평가: 검색된 문서 중 실제 정답 문서가 상위에 잘 정렬되었는지
    faithfulness,       # Generator 평가: 할루시네이션 방지 (주어진 문서에만 기반해 답했는지)
    answer_relevance    # Generator 평가: 사용자의 질문 의도에 맞게 답변했는지
)
from langchain_community.llms import HuggingFacePipeline
from langchain_core.retrievers import BaseRetriever

def evaluate_rag_system(
    test_dataset: List[Dict[str, Any]], 
    retriever: BaseRetriever, 
    rag_chain: Any,
    evaluation_llm: HuggingFacePipeline
) -> Dict[str, float]:
    """
    RAG 시스템의 검색 및 생성 성능을 Ragas 프레임워크로 종합 평가하는 함수.

    Args:
        test_dataset: [{"query": "질문", "ground_truth": "실제 정답"}] 형태의 리스트
        retriever: LangChain Retriever 객체 (FAISS + Reranker 파이프라인)
        rag_chain: 질문을 입력받아 최종 답변 문자열을 반환하는 LangChain 실행 체인
        evaluation_llm: 평가에 사용할 로컬 LLM (Qwen2) 객체
        
    Returns:
        각 평가지표별 평균 점수가 담긴 딕셔너리
    """
    
    # Ragas 데이터 변환용 빈 리스트 초기화
    queries = []
    contexts = []
    answers = []
    ground_truths = []
    
    print("🚀 RAG 파이프라인 데이터 수집 및 추론 시작...")
    
    for idx, item in enumerate(test_dataset):
        query = item.get("query")
        gt = item.get("ground_truth")
        
        # 1. 1차+2차 검색 파이프라인 통과 (Context 추출)
        retrieved_docs = retriever.invoke(query)
        # Ragas는 각 문서 내용을 문자열 리스트 포맷으로 요구함
        context_list = [doc.page_content for doc in retrieved_docs]
        
        # 2. LLM 추론 결과 획득 (Answer 생성)
        # 생성 결과물 문자열만 깔끔하게 추출하도록 연동 필요
        generated_answer = rag_chain.invoke({"context": "\n\n".join(context_list), "question": query})
        
        # 데이터셋 적재
        queries.append(query)
        contexts.append(context_list)
        answers.append(generated_answer)
        ground_truths.append(gt)
        
        print(f"   [{idx + 1}/{len(test_dataset)}] 전처리 완료 (질문: {query[:15]}...)")

    # 3. Ragas 입력용 HuggingFace Dataset 객체 생성
    data_dict = {
        "question": queries,
        "contexts": contexts,
        "answer": answers,
        "ground_truth": ground_truths
    }
    dataset = Dataset.from_dict(data_dict)
    
    # 4. 평가에 활용할 메트릭 및 평가 LLM 셋팅
    metrics = [context_recall, context_precision, faithfulness, answer_relevance]
    
    # Ragas 기본 매트릭스들이 내부 판별을 위해 주입받은 로컬 Qwen2 모델을 활용하도록 지정
    for metric in metrics:
        metric.llm = evaluation_llm
        
    print("\n📊 Ragas 평가지표 연산 시작 (LLM 심사 진행 중)...")
    
    # 5. Ragas 평가 실행
    result = evaluate(
        dataset=dataset,
        metrics=metrics
    )
    
    return dict(result)

# ------------------------------------------------------------------------
# 사용 예시 (이 모듈을 다른 파일에서 불러와 실행할 때의 스크립트 구조)
# ------------------------------------------------------------------------
if __name__ == "__main__":
    # 1. 사내 구축된 골든 데이터셋 준비 (질문 - 정답 셋)
    sample_test_set = [
        {
            "query": "이번 사내 전산망 고도화 사업의 장애 조치 SLA 기준은 어떻게 되나요?",
            "ground_truth": "장애 발생 시 30분 이내 원인 파악, 2시간 이내에 서비스를 정상 복구해야 하며, 이를 위반 시 페널티가 부과됩니다."
        },
        {
            "query": "RFP 14페이지에 명시된 데이터 백업 주기가 궁금합니다.",
            "ground_truth": "운영 데이터베이스는 매일 자정에 증분 백업을 수행하고, 매주 일요일에 전체 백업을 수행해야 합니다."
        }
    ]
    
    # 2. 기존 파이프라인에서 정의한 객체들 배치 (이전 단계의 변수들 가정)
    # final_retriever = compression_retriever (앞서 정의한 FAISS + 리랭커 조립체)
    # my_rag_chain = chain (Qwen2 프롬프트 템플릿 체인)
    # qwen_llm = llm (평가에 재사용할 로컬 Qwen2 인스턴스)
    
    # 3. 함수 호출 테스트 (구동환경에 맞게 인자 주입)
    # scores = evaluate_rag_system(
    #     test_dataset=sample_test_set,
    #     retriever=final_retriever,
    #     rag_chain=my_rag_chain,
    #     evaluation_llm=qwen_llm
    # )
    # print("\n=======  최종 Ragas 검증 결과 =======")
    # print(scores)